## Exercise: Notch Filter (2.X)

This exercise should be a design example of a recursive notch filter.  
Its effect is illustrated by some examples.

A 2nd order notch-filter is designed in the Z-domain.  
It has conjugate complex zeros on the unit circle — corresponding to frequency $\Omega_0$ that should be suppressed — and poles at the same phase angles that are marginally inside the unit circle.

**Zeros:** $e^{\pm j \Omega_0}$  
**Poles:** $\beta e^{\pm j \Omega_0}$

### a)
Write a Python function `notchdesign()` to calculate the filter coefficients  
$\mathbf{a} = \{a_0, a_1, a_2\}$ and $\mathbf{b} = \{b_0, b_1, b_2\}$.  
This function should use the normalized frequency $\Omega_0$ and $\beta$ as parameters.  
Calculate the coefficients either by analytical calculations or by using `np.poly()`.

### b)
Plot the frequency response (in dB) of a notch filter ($\Omega_N = \pi/4$, $\beta = 0.9$) using the function `scipy.signal.freqz()`.

What is the influence on the frequency response if you increase factor $\beta$ to $0.9999$?  
Verify your guess using Python.

### c)
This exercise demonstrates the effects of a notch filter by an audio example.

Create a test signal $x(k) = x_1(k) + x_2(k)$ that consists of the following components:

| Component | Description |
|------------|--------------|
| **1. sinus component** $x_1(k)$ | |
| frequency | $f_{sin} = 1000 \text{ Hz}$ |
| amplitude | $-0.5 \dots 0.5$ |
| **2. noise component** $x_2(k)$ | |
| distribution | equally distributed between $-0.25$ and $0.25$ |

Use a sampling frequency of $f_a = 8000 \text{ Hz}$.  
The test signal should span an interval of $5\,\mathrm{s}$.

Output the test signal $x(k)$ in Python using a playback library such as `sounddevice.play()`.  
Use a notch filter ($\beta = 0.9$) to suppress the sinusoidal signal $x_1(k)$ of $x(k)$.  
Only a noise signal should remain.

For filtering you may use the function `scipy.signal.lfilter()`.  
Output the filtered signal using `sounddevice.play()` to hear the effect of the notch filter.

### d)
Repeat the audio example of exercise (c).  
Use a notch filter with $\beta = 0.9999$.  
What has changed compared to the case of $\beta = 0.9$?  
What is the reason for this change?

Check the observed behavior by plotting the impulse responses of the notch filters with $\beta = 0.9$ and $\beta = 0.9999$.  
You can use `scipy.signal.lfilter()` and `np.arange()` to generate and plot the impulse responses.

### e) Additional exercise — practical example
If you receive some programs using your shortwave radio, you may notice distortions by sinusoidal signals.  
This might be carriers of other shortwave stations.

To demonstrate this behavior, there is the file  
`l936.28_kHz_25.04.2001_23_17.wav`.  
The distortion signal was measured at about $f = 345 \text{ Hz}$.

Now it’s your task to eliminate the distortion with some lines of Python code and output the cleaned signal through a loudspeaker.

**Hints:**
- Use Python’s functions `scipy.io.wavfile.read()` and `scipy.io.wavfile.write()` to handle the wave file.
- To fix $\beta$, select a value according to your experiences from exercises (b) and (d).
- Scale the filtered signal to have a maximum amplitude of 1.
- Depending on your system, you may need to use a separate program to play the output file if the sampling rate (e.g. $44.1 \text{ kHz}$) is not supported.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import lnotchdesign
import sounddevice as sd
# BEGIN SOLUTION
# ----- Part b: Dimensioning and Frequency Response ----------------------------
Omega0 = np.pi / 4

for beta in [0.9, 0.9999]:
    a, b = lnotchdesign(Omega0, beta)
    w, H = signal.freqz(b, a)

    plt.figure()
    plt.plot(w / np.pi, 20 * np.log10(np.maximum(np.abs(H), 1e-12)))
    plt.title(rf'Frequency Response $\beta$={beta}')
    plt.xlabel(r'$\Omega / \pi$')
    plt.ylabel(r'$|H(e^{j\Omega})|$ in dB')
    plt.ylim([-60, 10])
    plt.grid(True)

input("Continuing with Part c) — press Enter...")

# ----- Part c: Audio Demonstration --------------------------------------------
fa = 8000      # Sampling frequency
fsinus = 1000  # Sinusoid frequency
duration = 5   # seconds
N = int(fa * duration)

# Generate noisy sinusoidal signal
x1 = 0.5 * np.sin(2 * np.pi * fsinus / fa * np.arange(1, N + 1))
x2 = (np.random.rand(N) - 0.5) / 2.0
x = x1 + x2

print("Playing: x(k) unfiltered")
sd.play(x, samplerate=fa)
sd.wait()  # Wait for playback to finish

# Filter with beta = 0.9
beta = 0.9
Omega0 = 2 * np.pi * fsinus / fa
a, b = lnotchdesign(Omega0, beta)
y = signal.lfilter(b, a, x)

print(f"Playing: y(k) filtered beta={beta}")
sd.play(y, samplerate=fa)
sd.wait()

input("Continuing with Part d) — press Enter...")

# ----- Part d: Audio Demonstration, Impulse Response Length -------------------
print("Playing: x(k) unfiltered")
sd.play(x, samplerate=fa)
sd.wait()

beta = 0.9999
a, b = lnotchdesign(Omega0, beta)
y = signal.lfilter(b, a, x)

print(f"Playing: x(k) filtered beta={beta}")
sd.play(y, samplerate=fa)
sd.wait()

for beta in [0.9, 0.9999]:
    a, b = lnotchdesign(Omega0, beta)

    # Impulse response (N samples, sampling rate fa)
    imp = np.zeros(N, dtype=float)
    imp[0] = 1.0
    h = signal.lfilter(b, a, imp)
    t = np.arange(N) / fa

    # Downsample for plotting (every 10th sample)
    t_plot = t[::10]
    h_plot = h[::10]

    plt.figure()
    plt.stem(t_plot, h_plot)
    plt.axis([0, duration, -1e-4, 1e-4])
    plt.title(rf'Impulse Response $\beta$={beta}')
    plt.xlabel(r'$t = k\,T_A$ in s')
    plt.ylabel('h(k)')

plt.show()

# END SOLUTION

## Solution: Notch Filter (2.X)

### a)
Zeros: $e^{\pm j \Omega_N}$  
Poles: $\beta e^{\pm j \Omega_N}$

The transfer function $H(z)$ and therefore the filter coefficients are obtained through multiplication:

$$
\begin{array}{rcl}
H(z) & = & \frac{\left( z -  e^{+ j \Omega_N} \right) \left( z -  e^{- j \Omega_N} \right)}
{ \left( z - \beta e^{+ j \Omega_N} \right) \left( z - \beta e^{- j \Omega_N} \right)} \\
     & = &\frac{z^2 - z \left( e^{+ j \Omega_N} +  e^{- j \Omega_N} \right) + 1}
{z^2 - z \beta \left( e^{+ j \Omega_N} +  e^{- j \Omega_N} \right) + \beta^2} \\
     & = &\frac{z^2 - 2 z \cos\left( \Omega_N \right) + 1}
{z^2 - 2 \beta z \cos\left( \Omega_N \right) + \beta^2} \\
     & = &\frac{1 - 2 z^{-1} \cos\left( \Omega_N \right) + z^{-2}}
{1 - 2 \beta z^{-1} \cos\left( \Omega_N \right) + \beta^2 z^{-2}}
\end{array}
$$

With the normal definition of the coefficient pair $\mathbf{b}$ and $\mathbf{a}$

$$
H(z) = \frac{\sum\limits_{\mu=0}^{2} b_\mu z^{-\mu}}{1 + \sum\limits_{\nu=1}^{2} a_\nu z^{-\nu}}
$$

the values are:

$$
\mathbf{b} = \{1, -2\cos(\Omega_N), 1\}
$$

$$
\mathbf{a} = \{1, -2\beta\cos(\Omega_N), \beta^2\}
$$

### b)
It is clearly shown that the notch in the frequency domain becomes narrower if the poles (through factor $\beta$) are closer to the unit circle.

<img src="../pngs/ldsv2a19_b1.png" width="250">
<img src="../pngs/ldsv2a19_b2.png" width="250">

### c)
...listen and enjoy...

### d)
We hear that the notch filter with $\beta = 0.9999$ starts to affect the signal very slowly.  
This is because of the extremely long settling time of the filter.

The transient phenomenon is also reflected in the impulse response.

<img src="../pngs/ldsv2a19_d1.png" width="250">
<img src="../pngs/ldsv2a19_d2.png" width="250">

Explanation: Because the signal in the frequency domain is very narrow, the signal must be longer in the time domain.  
(Uncertainty principle of communications)

### e)
With appropriately chosen coefficients, the disturbance can be nearly eliminated.